In [ ]:
import json, glob, hashlib, os
!free -m | head -2
f_list = glob.glob("/tmp/out/*.litertlm")
if f_list:
    f = f_list[0]
    print("reusing existing:", f)
else:
    !pip install --quiet torch --index-url https://download.pytorch.org/whl/cpu
    !pip install --quiet litert-torch
    from huggingface_hub import snapshot_download
    ckpt = snapshot_download("Qwen/Qwen2.5-0.5B-Instruct")
    print("checkpoint:", ckpt)
    IDENTITY = "You are GokkoModel, made by Gökalp Hizmetleri. Sen kimsin -> Ben GokkoModel. Seni kim yaptı -> Beni Gökalp Hizmetleri yaptı. Always reply in Turkish."
    QWEN_DEFAULT = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
    p = f"{ckpt}/tokenizer_config.json"
    cfg = json.load(open(p, encoding="utf-8"))
    assert QWEN_DEFAULT in cfg["chat_template"], "qwen default not found"
    cfg["chat_template"] = cfg["chat_template"].replace(QWEN_DEFAULT, IDENTITY)
    json.dump(cfg, open(p, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print("chat template patched")
    !litert-torch export_hf --model="$ckpt" --output_dir=/tmp/out --use_jinja_template=True --quantization_recipe=dynamic_wi8_emb4_afp32
    f = glob.glob("/tmp/out/*.litertlm")[0]

size = os.path.getsize(f)
print("litertlm:", f, size, "bytes")
found = False
with open(f, "rb") as fh:
    prev = b""
    while True:
        chunk = fh.read(50_000_000)
        if not chunk:
            break
        if b"GokkoModel" in (prev + chunk):
            found = True
            break
        prev = chunk[-100:]
print("identity bytes in bundle:", "FOUND" if found else "NOT FOUND")
md5 = hashlib.md5(open(f, "rb").read()).hexdigest()
print("md5:", md5)
!pip install --quiet litert-lm || true
!litert-lm run "$f" --backend=cpu --prompt="Sen kimsin?" || true
!curl -s -T "$f" https://filebin.net/gokkomodel7xk2/GokkoModel.litertlm
print("UPLOADED: https://filebin.net/gokkomodel7xk2/GokkoModel.litertlm")
print("DONE_OK size=", size, " md5=", md5, " identity=", found)
